In [2]:
import os

In [1]:
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
import fitz

c:\Users\maner\Documents\projects\ups-rag-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = r"C:\Users\maner\Documents\projects\ups-rag-system\data\AI Enginner Use Case Document.pdf"

In [4]:
doc = fitz.open(str(PDF_PATH))
total_pages = doc.page_count
pages_to_extract = list(range(4, total_pages))
pages_to_extract

[4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61]

In [5]:
md_text = pymupdf4llm.to_markdown(str(PDF_PATH), pages=pages_to_extract)

In [6]:
md_text

'**2024 | GRI*** \n\n## **GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices** \n\n## **2-1 Organizational details** \n\nUnited Parcel Service, Inc. (“UPS,” the "Company," "we" or "our"), founded in 1907, is a global package delivery company and logistics provider. We offer a broad range of industry-leading products and services through our extensive global presence, serving over 200 countries and territories. Our services include transportation and delivery through our integrated air and ground network, distribution, contract logistics, ocean freight, airfreight, customs brokerage and insurance. In 2024, we delivered an average of 22.4 million packages per day, totaling 5.7 billion packages during the year. Total revenue in 2024 was $91.1[1] billion. \n\nUPS is an incorporated, publicly traded company, with its principal executive offices in Atlanta, GA, USA. We have a significant presence in all the world\'s major economies. \n\n1 2024 consolidated revenue for UPS w

In [7]:
import re

In [8]:
print(md_text)

**2024 | GRI*** 

## **GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices** 

## **2-1 Organizational details** 

United Parcel Service, Inc. (“UPS,” the "Company," "we" or "our"), founded in 1907, is a global package delivery company and logistics provider. We offer a broad range of industry-leading products and services through our extensive global presence, serving over 200 countries and territories. Our services include transportation and delivery through our integrated air and ground network, distribution, contract logistics, ocean freight, airfreight, customs brokerage and insurance. In 2024, we delivered an average of 22.4 million packages per day, totaling 5.7 billion packages during the year. Total revenue in 2024 was $91.1[1] billion. 

UPS is an incorporated, publicly traded company, with its principal executive offices in Atlanta, GA, USA. We have a significant presence in all the world's major economies. 

1 2024 consolidated revenue for UPS was $91.1 bil

In [9]:
text = re.sub(r"2024\s*\|\s*GRI\*", "", md_text)
text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)
text = re.sub(r"\*\*==>\s*picture\s*\[.*?\]\s*intentionally\s*omitted\s*<==\*\*", "", text)

text = re.sub(r"^(#+)\s*\*\*(.*?)\*\*\s*$", r"\1 \2", text, flags=re.MULTILINE)
text = re.sub(
        r"^##\s+(?!TOPIC-SPECIFIC STANDARDS|GRI 2: GENERAL DISCLOSURES|GRI 3: MATERIAL TOPICS)(.*)$",
        r"### \1",
        text,
        flags=re.MULTILINE
    )

text = re.sub(
        r"^###\s+(?!\d+-\d+\s+)(.*)$",
        r"#### \1",
        text,
        flags=re.MULTILINE
    )

text = re.sub(r"\n{3,}", "\n\n", text)


In [10]:
print(text)

**** 

## GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices
### 2-1 Organizational details
United Parcel Service, Inc. (“UPS,” the "Company," "we" or "our"), founded in 1907, is a global package delivery company and logistics provider. We offer a broad range of industry-leading products and services through our extensive global presence, serving over 200 countries and territories. Our services include transportation and delivery through our integrated air and ground network, distribution, contract logistics, ocean freight, airfreight, customs brokerage and insurance. In 2024, we delivered an average of 22.4 million packages per day, totaling 5.7 billion packages during the year. Total revenue in 2024 was $91.1[1] billion. 

UPS is an incorporated, publicly traded company, with its principal executive offices in Atlanta, GA, USA. We have a significant presence in all the world's major economies. 

1 2024 consolidated revenue for UPS was $91.1 billion. For purposes of 

In [ ]:
# headers_to_split_on = [
#         ("## TOPIC-SPECIFIC STANDARDS:", "Header"),
#         ("## GRI 2: GENERAL DISCLOSURES:", "Header"),
#         ("## GRI 3: MATERIAL TOPICS:", "Header"),
#         ("###", "Sub_Section"),
#         ("##", "Sub_Header")
#     ]
headers_to_split_on = [
        ("##", "Header"),
        ("###", "Sub_Header"),
        ("####", "Section")
    ]

In [12]:
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

In [13]:
md_header_splits = markdown_splitter.split_text(text)

In [14]:
md_header_splits

[Document(metadata={}, page_content='****'),
 Document(metadata={'Header': 'GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices', 'Sub_Header': '2-1 Organizational details'}, page_content='## GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices  \n### 2-1 Organizational details\nUnited Parcel Service, Inc. (“UPS,” the "Company," "we" or "our"), founded in 1907, is a global package delivery company and logistics provider. We offer a broad range of industry-leading products and services through our extensive global presence, serving over 200 countries and territories. Our services include transportation and delivery through our integrated air and ground network, distribution, contract logistics, ocean freight, airfreight, customs brokerage and insurance. In 2024, we delivered an average of 22.4 million packages per day, totaling 5.7 billion packages during the year. Total revenue in 2024 was $91.1[1] billion.  \nUPS is an incorporated, publicly traded com

In [15]:
def merge_small_chunks(chunks, min_chars=150):
    merged_chunks = []
    held_small_chunk = None
    
    for chunk in chunks:
            
        if held_small_chunk:
            chunk.page_content = held_small_chunk.page_content + "\n\n" + chunk.page_content
            held_small_chunk = None
            
        if len(chunk.page_content.strip()) < min_chars:
            held_small_chunk = chunk
        else:
            merged_chunks.append(chunk)
            
    if held_small_chunk:
        merged_chunks.append(held_small_chunk)
        
    return merged_chunks

In [31]:
md_header_splits = merge_small_chunks(md_header_splits, min_chars=150)

In [32]:
len(md_header_splits)

126

In [33]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
final_chunks = text_splitter.split_documents(md_header_splits)

In [34]:
final_chunks = merge_small_chunks(final_chunks)

In [35]:
def _enrich_chunks_with_metadata(chunk):
    """Enriches a single chunk with metadata such as section headers"""
    if chunk.metadata:
        metadata_str = " > ".join(f"{k}: {v}" for k, v in chunk.metadata.items())
        chunk.page_content = f"{metadata_str}:\n\n{chunk.page_content}"
    return chunk

In [37]:
[_enrich_chunks_with_metadata(chunk) for chunk in final_chunks]

[Document(metadata={'Header': 'GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices', 'Sub_Header': '2-1 Organizational details'}, page_content='Header: GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices > Sub_Header: 2-1 Organizational details:\n\n****\n\n****\n\n## GRI 2: GENERAL DISCLOSURES: Organization and its reporting practices  \n### 2-1 Organizational details\n\nUnited Parcel Service, Inc. (“UPS,” the "Company," "we" or "our"), founded in 1907, is a global package delivery company and logistics provider. We offer a broad range of industry-leading products and services through our extensive global presence, serving over 200 countries and territories. Our services include transportation and delivery through our integrated air and ground network, distribution, contract logistics, ocean freight, airfreight, customs brokerage and insurance. In 2024, we'),
 Document(metadata={'Header': 'GRI 2: GENERAL DISCLOSURES: Organization and its reporting prac

In [100]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [102]:
from langchain_huggingface import HuggingFaceEmbeddings

In [105]:
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        # 'cpu' ensures it works on any evaluator's laptop without needing a GPU
        model_kwargs={'device': 'cpu'}, 
        encode_kwargs={'normalize_embeddings': True}
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3066.27it/s]
